# Create Work Volumes (oxjob #1554)

Which book each Crossref chapter is in, and which proceedings volume each Crossref proceedings paper is in (#1528,
21 Nov 2026). A chapter is a work and its book is a work; `CreateWorksBase` puts the book on the chapter as
`volume {id, display_name}`, and `CreateWorkParents` turns it into `parent_works` with relationship `volume`.

```
openalex.crossref.crossref_deduplicated (raw: ISBNs, types)  --> crossref_volume_records
crossref_volume_records + location_work_ids                  --> work_volumes (one row per chapter location)
work_volumes                                                 --> CreateWorksBase.volume
```

Runs after `Map_Work_Ids` (both ends need work ids) and before `Works_Base`. Sizing and rule precision:
oxjobs `working/link-book-chapters-to-books/` (EXPLORE.md, DESIGN.md).

### Crossref book records and their parts, ISBNs normalized

In [ ]:
-- Crossref book records and their parts, with ISBNs normalized to ISBN-13 (oxjob #1554).
-- Walden's Crossref parse drops ISBNs, so they are read here from the raw feed. One row per Crossref record of a
-- book-ish type: containers (the books) and parts (chapters, sections, entries). Read by the next cell and by
-- CreateLocationsWithTypes (cr_type, title). oxjob #1528 (21 Nov 2026): proceedings volumes are containers and
-- proceedings papers are parts; container_title, publisher and year serve the proceedings title rule, and since
-- 22 Nov container_title2 (the second container title: the own title of the volume when the first names the series).
-- Also since 22 Nov, for minted proceedings volumes (cells below): issns (well-formed ISSNs, uppercased, sorted),
-- volume_number (the deposited volume with any Vol. prefix and leading zeros dropped) and event_name.
CREATE OR REPLACE TABLE openalex.works.crossref_volume_records
CLUSTER BY (doi)
AS
WITH base AS (
  SELECT
    LOWER(native_id) AS doi,
    type AS cr_type,
    GET(title, 0) AS title,
    GET(`container-title`, 0) AS container_title,
    GET(`container-title`, 1) AS container_title2,
    ARRAY_SORT(ARRAY_DISTINCT(FILTER(TRANSFORM(ISSN, x -> UPPER(TRIM(x))), x -> x RLIKE '^[0-9]{4}-[0-9]{3}[0-9X]$'))) AS issns,
    NULLIF(REGEXP_REPLACE(REGEXP_REPLACE(LOWER(TRIM(volume)), '^(vol(ume)?[.]?\\s*)', ''), '^0+(?=[0-9])', ''), '') AS volume_number,
    NULLIF(TRIM(event.name), '') AS event_name,
    publisher,
    GET(GET(issued.`date-parts`, 0), 0) AS year,
    ARRAY_DISTINCT(FILTER(TRANSFORM(ISBN, x -> REGEXP_REPLACE(UPPER(x), '[^0-9X]', '')), s -> LENGTH(s) IN (10, 13))) AS raw_isbns
  FROM openalex.crossref.crossref_deduplicated
  WHERE type IN (
    -- containers
    'book', 'monograph', 'edited-book', 'reference-book', 'book-set', 'proceedings',
    -- parts
    'book-chapter', 'book-section', 'book-part', 'book-track', 'reference-entry', 'other', 'proceedings-article'
  )
)
SELECT
  doi,
  cr_type,
  cr_type IN ('book', 'monograph', 'edited-book', 'reference-book', 'book-set', 'proceedings') AS is_container,
  title,
  container_title,
  container_title2,
  issns,
  volume_number,
  event_name,
  publisher,
  year,
  -- ISBN-10 -> ISBN-13: prefix 978, drop the old check digit, recompute the EAN-13 one
  ARRAY_DISTINCT(FILTER(TRANSFORM(raw_isbns, s -> CASE
    WHEN LENGTH(s) = 13 THEN s
    WHEN LENGTH(s) = 10 THEN CONCAT('978', SUBSTR(s, 1, 9), CAST((10 - AGGREGATE(SEQUENCE(1, 12), 0,
      (acc, i) -> acc + CAST(SUBSTR(CONCAT('978', SUBSTR(s, 1, 9)), i, 1) AS INT) * (CASE WHEN i % 2 = 1 THEN 1 ELSE 3 END)) % 10) % 10 AS STRING))
  END), s -> s RLIKE '^97[89][0-9]{10}$')) AS isbns
FROM base

### Link each part to its book's work

In [ ]:
-- work_volumes (oxjob #1554): which book each Crossref chapter is in, as OpenAlex work ids; since oxjob #1528 (21 Nov
-- 2026) also which proceedings volume each Crossref proceedings paper is in.
-- A part (chapter, section, entry, proceedings paper) links to a Crossref book or proceedings record by, in order of trust:
--   doi_stem     the part's DOI minus its last -_. segment, or cut at the first -_. after the slash, is a book record
--                (10.4324/9780429440229-7 -> 10.4324/9780429440229; 10.1007/978-3-642-40760-4_49 -> ...-4)
--   isbn         a part ISBN names exactly one book record, else exactly one with the part's DOI prefix
--   doi_pattern  Oxford Academic-hosted presses (...003.NNNN -> ...001.0001) and Peter Lang (10.3726/<isbn>/N)
--   container_title  proceedings papers only: the paper's container title and publisher equal a Crossref proceedings
--                record's title and publisher, that pair names exactly one proceedings record, and the two years are at
--                most one apart (a series title reused every year names one year's volume only)
--   container_title_2  the same with the paper's second container title, used only when no other rule names a volume:
--                series that deposit the series as container-title[0] and the volume's title as [1] (SEG, WIT, LREC)
-- When the stem and the ISBN name different books, no link (a set's or series' ISBN on the part, or a junk stem);
-- likewise when the container title names a different volume than a DOI rule.
-- Proceedings papers take a DOI stem only when the years are at most one apart, and never to a proceedings record that
-- papers from more than three years reach by stem: that is a series record, not a volume (10.5151/2594-5327).
-- Book parts never link to proceedings records, so their links are what they were before proceedings were added.
-- Both ends are mapped through location_work_ids, so the book is the work its own Crossref record lands on (that
-- settles book DOIs that two works share). One row per part location; CreateWorksBase picks the primary location's.
CREATE OR REPLACE TABLE openalex.works.work_volumes
CLUSTER BY (work_id)
AS
WITH containers AS (
  SELECT doi, cr_type, title, publisher, year, isbns FROM openalex.works.crossref_volume_records WHERE is_container
),
parts AS (
  SELECT doi, cr_type, isbns, LOWER(TRIM(container_title)) AS container_key,
    LOWER(TRIM(container_title2)) AS container_key2, publisher, year,
    REGEXP_EXTRACT(doi, '^(.*)[-_.][^-_./]+$', 1) AS stem1,
    REGEXP_EXTRACT(doi, '^(.*?)[-_.][^/]*$', 1) AS stem2,
    CASE
      WHEN doi RLIKE '\\.003\\.[0-9]{4}$' THEN REGEXP_REPLACE(doi, '\\.003\\.[0-9]{4}$', '.001.0001')
      WHEN doi RLIKE '^10\\.3726/[^/]+/[0-9]+$' THEN REGEXP_REPLACE(doi, '/[0-9]+$', '')
    END AS pattern_key
  FROM openalex.works.crossref_volume_records
  WHERE NOT is_container
),
isbn_index AS (
  SELECT isbn, COLLECT_SET(doi) AS book_dois
  FROM (SELECT doi, EXPLODE(isbns) AS isbn FROM containers WHERE cr_type <> 'proceedings')
  GROUP BY isbn
),
isbn_candidates AS (
  SELECT p.doi, ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(i.book_dois))) AS book_dois
  FROM (SELECT doi, EXPLODE(isbns) AS isbn FROM parts) p
  JOIN isbn_index i ON i.isbn = p.isbn
  GROUP BY p.doi
),
stem_series AS (  -- proceedings records that proceedings papers from more than three years reach by DOI stem
  SELECT COALESCE(s1.doi, s2.doi) AS book_doi
  FROM parts p
  LEFT JOIN containers s1 ON s1.doi = p.stem1 AND s1.cr_type = 'proceedings'
  LEFT JOIN containers s2 ON s2.doi = p.stem2 AND p.stem2 <> p.stem1 AND s2.cr_type = 'proceedings'
  WHERE p.cr_type = 'proceedings-article' AND COALESCE(s1.doi, s2.doi) IS NOT NULL
  GROUP BY 1
  HAVING COUNT(DISTINCT p.year) > 3
),
title_index AS (  -- proceedings records whose (title, publisher) names exactly one proceedings record
  SELECT LOWER(TRIM(title)) AS title_key, publisher, MIN(doi) AS book_doi, MIN(year) AS book_year
  FROM containers
  WHERE cr_type = 'proceedings' AND TRIM(COALESCE(title, '')) <> '' AND publisher IS NOT NULL
  GROUP BY 1, 2
  HAVING COUNT(*) = 1
),
picks AS (
  SELECT
    p.doi,
    CASE
      WHEN p.cr_type = 'proceedings-article' AND (ss.book_doi IS NOT NULL
        OR ABS(p.year - CASE WHEN s1.doi IS NOT NULL THEN s1.year ELSE s2.year END) > 1) THEN NULL
      ELSE COALESCE(s1.doi, s2.doi)
    END AS stem_book,
    CASE
      WHEN SIZE(c.book_dois) = 1 THEN c.book_dois[0]
      WHEN SIZE(FILTER(c.book_dois, b -> SUBSTRING_INDEX(b, '/', 1) = SUBSTRING_INDEX(p.doi, '/', 1))) = 1
        THEN FILTER(c.book_dois, b -> SUBSTRING_INDEX(b, '/', 1) = SUBSTRING_INDEX(p.doi, '/', 1))[0]
    END AS isbn_book,
    pt.doi AS pattern_book,
    CASE WHEN p.cr_type = 'proceedings-article' AND ABS(p.year - ti.book_year) <= 1 THEN ti.book_doi END AS title_book1,
    CASE WHEN p.cr_type = 'proceedings-article' AND ABS(p.year - ti2.book_year) <= 1 THEN ti2.book_doi END AS title_book2
  FROM parts p
  LEFT JOIN containers s1 ON s1.doi = p.stem1 AND (p.cr_type = 'proceedings-article' OR s1.cr_type <> 'proceedings')
  LEFT JOIN containers s2 ON s2.doi = p.stem2 AND p.stem2 <> p.stem1
    AND (p.cr_type = 'proceedings-article' OR s2.cr_type <> 'proceedings')
  LEFT JOIN stem_series ss ON ss.book_doi = COALESCE(s1.doi, s2.doi)
  LEFT JOIN isbn_candidates c ON c.doi = p.doi
  LEFT JOIN containers pt ON pt.doi = p.pattern_key AND (p.cr_type = 'proceedings-article' OR pt.cr_type <> 'proceedings')
  LEFT JOIN title_index ti ON ti.title_key = p.container_key AND ti.publisher = p.publisher
  LEFT JOIN title_index ti2 ON ti2.title_key = p.container_key2 AND ti2.publisher = p.publisher
),
picks2 AS (
  SELECT *, COALESCE(title_book1,
    CASE WHEN COALESCE(stem_book, isbn_book, pattern_book) IS NULL THEN title_book2 END) AS title_book
  FROM picks
),
links AS (
  SELECT doi,
    COALESCE(stem_book, isbn_book, pattern_book, title_book) AS book_doi,
    CASE
      WHEN stem_book IS NOT NULL AND isbn_book IS NOT NULL THEN 'isbn_and_doi_stem'
      WHEN stem_book IS NOT NULL THEN 'doi_stem'
      WHEN isbn_book IS NOT NULL THEN 'isbn'
      WHEN pattern_book IS NOT NULL THEN 'doi_pattern'
      WHEN title_book1 IS NOT NULL THEN 'container_title'
      ELSE 'container_title_2'
    END AS method
  FROM picks2
  WHERE COALESCE(stem_book, isbn_book, pattern_book, title_book) IS NOT NULL
    AND NOT (stem_book IS NOT NULL AND isbn_book IS NOT NULL AND stem_book <> isbn_book)
    AND NOT (title_book IS NOT NULL AND COALESCE(stem_book, isbn_book, pattern_book) IS NOT NULL
             AND title_book <> COALESCE(stem_book, isbn_book, pattern_book))
)
SELECT
  part.work_id,
  l.doi AS native_id,
  book.work_id AS volume_work_id,
  l.book_doi AS volume_doi,
  c.title AS volume_display_name,
  l.method
FROM links l
JOIN openalex.works.location_work_ids part
  ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = l.doi
JOIN openalex.works.location_work_ids book
  ON book.provenance = 'crossref' AND book.native_id_namespace = 'doi' AND book.native_id = l.book_doi
JOIN containers c ON c.doi = l.book_doi
WHERE part.work_id IS NOT NULL AND book.work_id IS NOT NULL AND part.work_id <> book.work_id

### Minted proceedings volumes (oxjob #1528, phase 1)

Crossref has no record for the volume of ~3.4M proceedings papers. Phase 1 mints one `book` work per volume that the
papers name by two permanent identifiers, the series ISSN and the volume number (Proc. SPIE 10000, AIP Conference
Proceedings 2220). The registry `openalex.works.minted_volumes` keeps every volume once minted, so a threshold that
drifts never deletes one; `CreateLocationsWithTypes` writes one location per registry row (provenance `openalex-series`,
namespace `volume`) the next night, `Map_Work_Ids` gives it a W id, and the last cell links the papers to it.

Registry `status`: `active` = minted and linked; `unlinked` = minted (the W id lives) but no papers link to it (use it
to pull links back without deleting ids); `excluded` = never minted (deposit errors found by the bar).

In [0]:
-- oxjob #1528 (22 Nov 2026): the registry of minted proceedings volumes, one row per (series, volume number). Rows are
-- only added or refreshed, never deleted: the W id of a minted volume must not die when a threshold below drifts.
CREATE TABLE IF NOT EXISTS openalex.works.minted_volumes (
  mint_id STRING COMMENT 'proceedings:<series issn, lowercased>:<volume number>, the native_id of the minted location',
  issn STRING COMMENT 'the series: its ISSN-L when the source registry knows it, else its lowest ISSN, fixed at minting',
  series_issns ARRAY<STRING> COMMENT 'every ISSN seen on the series papers (only grows), so a paper matches whichever it carries',
  volume_number STRING,
  title STRING,
  year INT COMMENT 'the papers modal year',
  published_date DATE COMMENT 'earliest paper date in the modal year',
  publisher STRING,
  source_id BIGINT COMMENT 'the series source the papers sit under',
  status STRING COMMENT 'active, unlinked or excluded (see the notebook text)',
  papers BIGINT COMMENT 'qualifying papers last time the volume was assessed',
  first_seen DATE,
  last_seen DATE,
  updated_at DATE COMMENT 'last day the title, year, date, publisher or source changed'
) CLUSTER BY (mint_id)

In [0]:
-- oxjob #1528 (22 Nov 2026), minted proceedings volumes, phase 1: which volumes qualify tonight. Two outside reviews
-- (GPT, Gemini, 22 Nov) shaped these rules:
--   series    the series of a paper is the one ISSN-L its ISSNs map to in the source registry, else its lowest ISSN. A series
--             qualifies when none of its proceedings papers links to a Crossref volume record by the rules above and
--             >= 90% sit under a conference or book-series source (journal supplements stay out)
--   numbering a series that reuses volume numbers across years (> 5% of its numbers span > 2 years) is out, and a volume
--             number must hold a digit
--   volume    >= 2 papers, >= 95% of them within the modal year +-1 (the others are not linked), a non-empty title
--   title     the second container title when it names one volume number of the series, else the one event name when
--             it does, else <series>, Volume N. Every pick is the most frequent value, ties to the larger, so reruns agree
--   new only  a volume is new when no registry row has its volume number and shares an ISSN with it, and it is minted
--             only if no work OpenAlex already has carries its title and year +-1 (no duplicate). Registry rows are
--             refreshed while they still qualify and kept whatever tonight says.
-- Bar 22 Nov 2026 (oxjob #1528 PLAN.md): papers 148/150 (SPIE 50, AIP 49, other 49), volumes 99/100.
MERGE INTO openalex.works.minted_volumes r
USING (
  WITH linked AS (SELECT DISTINCT native_id FROM openalex.works.work_volumes WHERE COALESCE(method, '') <> 'minted'),
  issn_map AS (
    SELECT issn, MIN(issn_l) AS issn_l
    FROM (SELECT UPPER(TRIM(i)) AS issn, UPPER(TRIM(issn_l)) AS issn_l
          FROM openalex.sources.sources LATERAL VIEW EXPLODE(issns) e AS i WHERE issn_l IS NOT NULL)
    GROUP BY issn HAVING COUNT(DISTINCT issn_l) = 1
  ),
  loc AS (
    SELECT LOWER(native_id) AS doi, MAX(source_id) AS source_id, MIN(published_date) AS published_date
    FROM openalex.works.locations_w_types
    WHERE provenance = 'crossref' AND native_id_namespace = 'doi'
    GROUP BY 1
  ),
  p0 AS (
    SELECT c.doi, c.issns, c.volume_number AS vnum, c.year, c.publisher,
      NULLIF(TRIM(c.container_title), '') AS container_title, NULLIF(TRIM(c.container_title2), '') AS container_title2,
      c.event_name, l.source_id, l.published_date, s.type AS source_type, k.native_id IS NOT NULL AS is_linked
    FROM openalex.works.crossref_volume_records c
    LEFT JOIN linked k ON k.native_id = c.doi
    LEFT JOIN loc l ON l.doi = c.doi
    LEFT JOIN openalex.sources.sources s ON s.id = l.source_id
    WHERE c.cr_type = 'proceedings-article' AND SIZE(c.issns) > 0
  ),
  pm AS (
    SELECT x.doi, COLLECT_SET(m.issn_l) AS ls
    FROM (SELECT doi, EXPLODE(issns) AS i FROM p0) x LEFT JOIN issn_map m ON m.issn = x.i
    GROUP BY x.doi
  ),
  papers AS (
    SELECT p0.*, CASE WHEN SIZE(pm.ls) = 1 THEN pm.ls[0] ELSE ARRAY_MIN(p0.issns) END AS issn
    FROM p0 JOIN pm USING (doi)
  ),
  series AS (
    SELECT issn, ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(issns)))) AS series_issns
    FROM papers GROUP BY issn
    HAVING COUNT_IF(is_linked) = 0 AND COUNT_IF(source_type IN ('conference', 'book series')) >= 0.9 * COUNT(*)
  ),
  cand AS (
    SELECT p.*, s.series_issns FROM papers p JOIN series s USING (issn)
    WHERE p.vnum RLIKE '^[0-9a-z][0-9a-z.-]{0,19}$' AND p.vnum RLIKE '[0-9]'
      AND p.year BETWEEN 1900 AND YEAR(CURRENT_DATE()) + 1
  ),
  reuse AS (
    SELECT issn FROM (SELECT issn, vnum, MAX(year) - MIN(year) AS span FROM cand GROUP BY 1, 2)
    GROUP BY issn HAVING SUM(CASE WHEN span > 2 THEN 1 ELSE 0 END) > 0.05 * COUNT(*)
  ),
  c2 AS (SELECT cand.* FROM cand LEFT ANTI JOIN reuse USING (issn)),
  ycount AS (SELECT issn, vnum, year, COUNT(*) AS k FROM c2 GROUP BY 1, 2, 3),
  modal AS (SELECT issn, vnum, MAX_BY(year, k * 10000 + year) AS my FROM ycount GROUP BY 1, 2),
  c3 AS (SELECT c2.*, m.my FROM c2 JOIN modal m USING (issn, vnum)),
  vt AS (
    SELECT issn, LOWER(container_title2) AS k FROM c3 WHERE container_title2 IS NOT NULL
    GROUP BY 1, 2 HAVING COUNT(DISTINCT vnum) = 1
  ),
  ev AS (
    SELECT issn, LOWER(event_name) AS k FROM c3 WHERE event_name IS NOT NULL
    GROUP BY 1, 2 HAVING COUNT(DISTINCT vnum) = 1
  ),
  c4 AS (
    SELECT c3.*, CASE WHEN vt.k IS NOT NULL THEN c3.container_title2 END AS vtitle,
      CASE WHEN ev.k IS NOT NULL THEN c3.event_name END AS etitle
    FROM c3
    LEFT JOIN vt ON vt.issn = c3.issn AND vt.k = LOWER(c3.container_title2)
    LEFT JOIN ev ON ev.issn = c3.issn AND ev.k = LOWER(c3.event_name)
  ),
  c5 AS (
    SELECT c4.*,
      COUNT(publisher) OVER (PARTITION BY issn, vnum, publisher) AS n_pub,
      COUNT(source_id) OVER (PARTITION BY issn, vnum, source_id) AS n_src,
      COUNT(container_title) OVER (PARTITION BY issn, vnum, container_title) AS n_ser,
      COUNT(vtitle) OVER (PARTITION BY issn, vnum, vtitle) AS n_vt,
      COUNT(etitle) OVER (PARTITION BY issn, vnum, etitle) AS n_et
    FROM c4
  ),
  vols AS (
    SELECT issn, vnum, MAX(series_issns) AS series_issns, COUNT(*) AS n, COUNT_IF(ABS(year - my) <= 1) AS n_in,
      MAX(my) AS year, MIN(CASE WHEN YEAR(published_date) = my THEN published_date END) AS published_date,
      MAX_BY(publisher, STRUCT(n_pub, publisher)) AS publisher,
      MAX_BY(source_id, STRUCT(n_src, source_id)) AS source_id,
      MAX_BY(container_title, STRUCT(n_ser, container_title)) AS series_title,
      MAX_BY(vtitle, STRUCT(n_vt, vtitle)) AS volume_title,
      COUNT(DISTINCT LOWER(etitle)) AS n_events,
      MAX_BY(etitle, STRUCT(n_et, etitle)) AS event_title
    FROM c5 GROUP BY issn, vnum
  ),
  ok AS (
    SELECT * FROM (
      SELECT CONCAT('proceedings:', LOWER(issn), ':', vnum) AS mint_id, issn, series_issns, vnum AS volume_number,
        NULLIF(TRIM(COALESCE(volume_title, CASE WHEN n_events = 1 THEN event_title END,
          CASE WHEN series_title IS NOT NULL THEN CONCAT(series_title, ', Volume ', vnum) END)), '') AS title,
        CAST(year AS INT) AS year, COALESCE(published_date, MAKE_DATE(year, 1, 1)) AS published_date,
        publisher, source_id, n_in AS papers
      FROM vols WHERE n_in >= 2 AND n_in >= 0.95 * n
    ) WHERE title IS NOT NULL
  ),
  fresh AS (
    SELECT ok.mint_id FROM ok LEFT ANTI JOIN openalex.works.minted_volumes m
      ON m.volume_number = ok.volume_number AND ARRAYS_OVERLAP(m.series_issns, ok.series_issns)
  ),
  dup AS (
    SELECT DISTINCT ok.mint_id FROM ok JOIN fresh USING (mint_id) JOIN openalex.works.openalex_works w
      ON LOWER(TRIM(w.title)) = LOWER(ok.title) AND ABS(w.publication_year - ok.year) <= 1
     AND w.type IN ('book', 'paratext', 'other', 'conference-paper', 'report')
  )
  SELECT ok.*, fresh.mint_id IS NOT NULL AS is_fresh, dup.mint_id IS NOT NULL AS is_dup
  FROM ok LEFT JOIN fresh USING (mint_id) LEFT JOIN dup USING (mint_id)
) s
ON r.mint_id = s.mint_id
WHEN MATCHED AND NOT (r.title <=> s.title AND r.year <=> s.year AND r.published_date <=> s.published_date
    AND r.publisher <=> s.publisher AND r.source_id <=> s.source_id) THEN UPDATE SET
  title = s.title, year = s.year, published_date = s.published_date, publisher = s.publisher, source_id = s.source_id,
  series_issns = ARRAY_SORT(ARRAY_DISTINCT(CONCAT(COALESCE(r.series_issns, ARRAY()), s.series_issns))), papers = s.papers,
  last_seen = CURRENT_DATE(), updated_at = CURRENT_DATE()
WHEN MATCHED THEN UPDATE SET
  series_issns = ARRAY_SORT(ARRAY_DISTINCT(CONCAT(COALESCE(r.series_issns, ARRAY()), s.series_issns))), papers = s.papers,
  last_seen = CURRENT_DATE()
WHEN NOT MATCHED AND s.is_fresh AND NOT s.is_dup THEN INSERT (mint_id, issn, series_issns, volume_number, title, year,
  published_date, publisher, source_id, status, papers, first_seen, last_seen, updated_at)
  VALUES (s.mint_id, s.issn, s.series_issns, s.volume_number, s.title, s.year, s.published_date, s.publisher,
  s.source_id, 'active', s.papers, CURRENT_DATE(), CURRENT_DATE(), CURRENT_DATE())

In [0]:
-- oxjob #1528 (22 Nov 2026): proceedings papers that no rule above links take their minted volume (method `minted`):
-- an `active` registry row with the volume number of the paper that shares an ISSN with it, year within 1 of the volume
-- year, and exactly one such row. A work keeps any volume it already has through one of its DOIs, and a work whose DOIs
-- name two different minted volumes takes neither. The W id of the volume comes from its minted location
-- (CreateLocationsWithTypes, provenance `openalex-series`, namespace `volume`), so a volume first registered tonight
-- gets its location, and so its papers, the next night.
INSERT INTO openalex.works.work_volumes
WITH m AS (
  SELECT c.doi, MIN(r.mint_id) AS mint_id, MIN(r.title) AS title
  FROM openalex.works.crossref_volume_records c
  JOIN openalex.works.minted_volumes r
    ON r.status = 'active' AND r.volume_number = c.volume_number AND ARRAYS_OVERLAP(r.series_issns, c.issns)
   AND ABS(c.year - r.year) <= 1
  WHERE c.cr_type = 'proceedings-article'
  GROUP BY c.doi
  HAVING COUNT(DISTINCT r.mint_id) = 1
),
p AS (
  SELECT part.work_id, m.doi, m.mint_id, m.title
  FROM m JOIN openalex.works.location_work_ids part
    ON part.provenance = 'crossref' AND part.native_id_namespace = 'doi' AND part.native_id = m.doi
  WHERE part.work_id IS NOT NULL
),
one AS (SELECT work_id FROM p GROUP BY work_id HAVING COUNT(DISTINCT mint_id) = 1)
SELECT p.work_id, p.doi AS native_id, vol.work_id AS volume_work_id, CAST(NULL AS STRING) AS volume_doi,
  p.title AS volume_display_name, 'minted' AS method
FROM p
JOIN one USING (work_id)
JOIN openalex.works.location_work_ids vol
  ON vol.provenance = 'openalex-series' AND vol.native_id_namespace = 'volume' AND vol.native_id = p.mint_id
LEFT ANTI JOIN (SELECT DISTINCT work_id FROM openalex.works.work_volumes) w ON w.work_id = p.work_id
WHERE vol.work_id IS NOT NULL AND p.work_id <> vol.work_id